In [162]:
import json
import pandas as pd
import shopify
import datetime
import requests
import pytz
import os
import binascii
import time
import string
import random
import re
import json
import warnings
import warnings
warnings.filterwarnings("ignore")

In [147]:
# Read secrets
with open(r'D:\\Study\\THConsultant\\secret.json', 'r') as jfile:
    secrets = json.load(jfile)
shop_name = secrets['shop_name']
token = secrets['token']
api_key = secrets['api_key']
api_password = secrets['api_password']
shopify_store_domain = secrets['shopify_store_domain']
api_version = secrets['api_version']   
    
# Shopify settings
orders_endpoint = f'https://{shopify_store_domain}/admin/api/{api_version}/orders.json?status=any'
shop_url = "https://%s:%s@%s.myshopify.com/admin" % (api_key, token, shop_name)
shopify.ShopifyResource.set_site(shop_url)
shopify.Session.setup(api_key=api_key, secret=token)

In [148]:
# new_customer = shopify.Customer.create({
#     "first_name": 'Dunj 5',
#     "last_name": 'For test',
#     "email": 'test_2@gmail.com',
#     "verified_email": True,
#     "addresses": [{
#         "address1": "123 Fake Street",
#         "city": "Faketown",
#         "province": "ON",
#         "phone": "555-555-5555",
#         "zip": "12345",
#         "last_name": 'Cust',
#         "first_name": 'Test 2',
#         "country": "CA"
#     }]
# })
# new_customer.save()

In [149]:
pattern = re.compile(r'Dunj', re.IGNORECASE)
customers = shopify.Customer.find()
for cust in customers:
    first_name = cust.first_name if cust.first_name else ""
    if re.search(pattern, first_name):
        print(cust.id)
        # shopify.Customer.find(cust.id).destroy()

8033810514247
8033658241351
8033657291079


In [369]:
cust_ids = [8033658241351, 8033657291079]

In [380]:
import os
from email.message import EmailMessage
import ssl
import smtplib
code = 'axvx ovsb hffd dpls'
email_sender = 'dam0709123@gmail.com'
password = 'axvx ovsb hffd dpls'
email_receiver = 'skilerVoi@gmail.com'

subject = 'Check out'
body = """

hello mother fucker
"""
em = EmailMessage()
em['From'] = email_sender
em['To'] = email_receiver
em['Subject'] = subject
em.set_content(body)

context = ssl.create_default_context()
with smtplib.SMTP_SSL('smtp.gmail.com', 465, context= context) as smtp:
    smtp.login(email_sender, password)
    smtp.sendmail(email_sender, email_receiver, em.as_string())


# Promotion Code 

In [6]:
def generate_discount_code(id):
    random_str = ''.join(random.choices(string.ascii_uppercase, k=2))
    timestamp = int(time.time())
    return f'PRM{id % 1000}{timestamp % 10000}{random_str}'


def create_prom_code(code, cust_id):
    defaults = {
        "title": f"{code}",
        "target_type": "line_item",
        "target_selection": "all",
        "allocation_method": "across",
        "value_type": "percentage",  # or "fixed_amount"
        "value": -10.0,  # -10 percent discount
        "customer_selection": "prerequisite",
        "starts_at": datetime.datetime.now().isoformat(),
        "ends_at": (datetime.datetime.now() + datetime.timedelta(days=30)).isoformat(), 
        "prerequisite_customer_ids": [cust_id], # main is top5['ID']
        "usage_limit": 1,  # The maximum number of times the price rule can be used, per discount code
        'once_per_customer': True
    }
    price_rule = shopify.PriceRule.create(defaults)
    if price_rule.save():
        print(f"Price rule created successfully ")
    else:
        print("Failed to create price rule.")
    # Create Discount code
    discount_code = shopify.DiscountCode.create({
        "price_rule_id": price_rule.id,
        "code": code
    })
    if discount_code.save():
        print(f"Discount code created successfully")
    else:
        print("Failed to create Discount code.")
    
    return price_rule.id, discount_code.id

def create_prom_codes(cust_ids):
    prs = []
    dcs = []
    for cust_id in cust_ids:
        code = generate_discount_code(cust_id)
        pr, dc = create_prom_code(code, cust_id)
        prs.append(pr)
        dcs.append(dc)
        print(f'{cust_id} - {code}')
    return prs, dcs

def delete_discount_code(pr_id):
    dcc = shopify.DiscountCode.find(price_rule_id=pr_id)
    for c in dcc:
        shopify.DiscountCode.find(id_=c.id, price_rule_id=pr_id).destroy()
    shopify.PriceRule.find(pr_id).destroy()
    print('Delete Finished')


In [7]:
prs, dcs = create_prom_codes(cust_ids)

Price rule created successfully 
Discount code created successfully
8033658241351 - PRM3518393SU
Price rule created successfully 
Discount code created successfully
8033657291079 - PRM798396CV


In [8]:
for pr in prs:
    delete_discount_code(pr)

# Reference Code 

In [16]:
def generate_discount_code(id):
    random_str = ''.join(random.choices(string.ascii_uppercase, k=2))
    timestamp = int(time.time())
    return f'REFE{id % 1000}{timestamp % 10000}{random_str}'


def create_refe_code(code, seg_id, cust_id):
    defaults = {
        "title": f"{code}",
        "target_type": "line_item",
        "target_selection": "all",
        "allocation_method": "across",
        "value_type": "percentage",  # or "fixed_amount"
        "value": -10.0,  # -10 percent discount
        "customer_selection": "prerequisite",
        "starts_at": datetime.datetime.now().isoformat(),
        "ends_at": (datetime.datetime.now() + datetime.timedelta(days=30)).isoformat(), 
        "customer_segment_ids": [cust_id],
        "customer_segment_prerequisite_ids": [seg_id],
        "usage_limit": 10,  
        'once_per_customer': True
    }
    price_rule = shopify.PriceRule.create(defaults)
    if price_rule.save():
        print(f"Price rule created successfully ")
    else:
        print("Failed to create price rule.")
    # Create Discount code
    discount_code = shopify.DiscountCode.create({
        "price_rule_id": price_rule.id,
        "code": code
    })
    if discount_code.save():
        print(f"Discount code created successfully")
    else:
        print("Failed to create Discount code.")
    
    return price_rule.id, discount_code.id

def create_refe_codes(cust_ids, seg_id):
    prs = []
    dcs = []
    codes = []
    for cust_id in cust_ids:
        code = generate_discount_code(cust_id)
        pr, dc = create_refe_code(code, seg_id, cust_id)
        prs.append(pr)
        dcs.append(dc)
        codes.append(code)
        print(f'{cust_id} - {code}')
    return prs, dcs, codes

def delete_discount_code(pr_id):
    dcc = shopify.DiscountCode.find(price_rule_id=pr_id)
    for c in dcc:
        shopify.DiscountCode.find(id_=c.id, price_rule_id=pr_id).destroy()
    shopify.PriceRule.find(pr_id).destroy()
    print('Delete Finished')



In [17]:
query = """
{
  segments(first: 10) {
    edges {
      node {
        id
        name
      }
    }
  }
}
"""


url = f'https://{shopify_store_domain}/admin/api/{api_version}/graphql.json'
headers = {"Content-Type": "application/graphql",
           "X-Shopify-Access-Token": token}

request = requests.post(url, data=query, headers=headers)
load = json.loads(request.text)
for seg in load['data']['segments']['edges']:
    id_match = re.search(r'\d+', seg['node']['id']).group()
    print(f"{seg['node']['name']} - {id_match}")


Abandoned checkouts in the last 30 days - 409831932083
Email subscribers - 409831964851
Customers who haven't purchased - 409831997619
Customers who have purchased more than once - 409832030387
Customers who have purchased at least once - 1026358313287


In [78]:
prs, dcs, codes = create_refe_codes(cust_ids, 409831997619)

Price rule created successfully 
Discount code created successfully
8033658241351 - REFE3512997FV
Price rule created successfully 
Discount code created successfully
8033657291079 - REFE792999CH


In [77]:
for pr in prs:
    delete_discount_code(pr)

Delete Finished
Delete Finished


In [251]:
def update_used_code(prs, dcs):
    usage_count = []
    usage_limit = []
    for i in range(len(prs)):
        usage_limit.append(shopify.PriceRule.find(prs[i]).usage_limit)
        usage_count.append(shopify.DiscountCode.find(id_= dcs[i], price_rule_id= prs[i]).usage_count)
    return usage_count, usage_limit

def compare_lists(list1, list2):
    indices = []
    values = []
    for i, (a, b) in enumerate(zip(list1, list2)):
        value = a - b
        if value != 0:
            indices.append(i)
            values.append(value)
    return indices, values

In [265]:
# usage_count,usage_limit = update_used_code(cust_ids, prs, dcs)
# refers_code = pd.DataFrame(list(zip(cust_ids, prs, dcs, codes, usage_count, usage_limit)), columns= ['ID', 'PriceRule_ID', 'DiscountCode_ID', 'Refer_Code', 'Usage Count','Usage Limit'])
# refers_code['Update at'] = datetime.datetime.now().strftime("%d/%m/%Y")
# refers_code['Create at'] = datetime.datetime.now().strftime("%d/%m/%Y")
# refers_code.to_csv('Current_Version.csv')

# old_ver = pd.DataFrame(columns= refers_code.columns)
# old_ver.to_csv('Old_version.csv')


In [364]:
def check_update(prs, dcs, cur_ver):
    usage_count, usage_limit = update_used_code(prs, dcs)
    # usage_count= [0,3]
    # usage_limit= [10, 10]
    update_time = (datetime.datetime.now()).strftime("%d/%m/%Y")
    dif_inds, values = compare_lists(usage_count, cur_ver['Usage Count'])
    # update_time = (datetime.datetime.now()+datetime.timedelta(days=1)).strftime("%d/%m/%Y")
    if dif_inds:
        return dif_inds, values, usage_count, update_time 
    else: 
        print('Nothing Changed')
        return None, None, None, None

def update(old_ver, new_ver, dif_inds, usage_count, update_time):
    if dif_inds and usage_count and update_time:  
        old_ver = old_ver._append(new_ver.iloc[dif_inds,:].copy())
        new_ver['Usage Count'] = usage_count  
        new_ver['Update at'] = update_time
        print("Update Finished")
        return old_ver, new_ver
    else:
        print('Not update') 
        return None, None

def get_reward(cust_ids, values, rew_code_cust):
    prs = []
    dcs = []
    for i,cust in enumerate(cust_ids):
        if cust in rew_code_cust:
            price_rule = shopify.PriceRule.find(rew_code_cust[cust])
            price_rule.value = float(price_rule.value) + (values[i])*random.randint(-10,-5)
            price_rule.save()
            print(price_rule.id)

        else:
            random_str = ''.join(random.choices(string.ascii_uppercase, k=2))
            timestamp = int(time.time())
            code = f'REW{cust%10}{cust % 100}{timestamp % 10000}{random_str}'
            defaults = {
            "title": f"{code}",
            "target_type": "line_item",
            "target_selection": "all",
            "allocation_method": "across",
            "value_type": "fixed_amount",
            "value": random.randint(-10,-5),  
            "customer_selection": "prerequisite",
            "starts_at": datetime.datetime.now().isoformat(),
            "ends_at": (datetime.datetime.now() + datetime.timedelta(days=30)).isoformat(), 
            "prerequisite_customer_ids": [cust],
            "usage_limit": None,  
            'once_per_customer': False,
            "combine_with_other_discount_codes": True
            }
            try:
                price_rule = shopify.PriceRule.create(defaults)
                price_rule.save()
                discount_code = shopify.DiscountCode.create({
                    "price_rule_id": price_rule.id,
                    "code": code
                })
                discount_code.save()
                prs.append(price_rule.id)
                dcs.append(discount_code.id)
                print(f'{cust} - {code}')
            except Exception as e:
                print(f'Error creating discount for {cust}')
            for i in values:
                if i > 1:
                    price_rule = shopify.PriceRule.find(price_rule.id)
                    price_rule.value = float(price_rule.value) + (i-1)*random.randint(-10,-5)
                    price_rule.save()
                else:
                    continue
            rew_code_cust[cust] = price_rule.id
    return prs, dcs, rew_code_cust

In [361]:
rewarded_cust = {} # only use 1 times

In [362]:
old_ver = pd.read_csv('Old_version.csv', index_col= 0)
cur_ver = pd.read_csv('Current_Version.csv', index_col=0)
dif_inds, values, usage_count, update_time = check_update(prs, dcs, cur_ver)
old_ver,new_ver = update(old_ver, cur_ver, dif_inds, usage_count, update_time)
if not new_ver.empty:
    prs, dcs, rewarded_cust = get_reward(new_ver.iloc[dif_inds,0], values, rewarded_cust)
else: 
    print('No Rewards Available')
# old_ver.to_csv('Old_version.csv')
# new_ver.to_csv('Current_Version.csv')

Update Finished
1686109290823


# Real Cases

In [13]:
data = pd.read_csv('D:\Study\THConsultant\Data.csv')
# top 5 most profitable clients 
top10 = data.groupby(['Client name', 'ID'])[['Quantity','Revenue']].sum().sort_values(['Revenue','Quantity'], ascending= False)[:10].reset_index()
top10

,Client name,ID,Quantity,Revenue
0,Slaczka Marcin,6357157707955,11,1100.14
1,Kempe Sara,8030427578695,6,930.63
2,Farrelly Rachel,6378218750131,1,929.00
3,Blake Marina,7960178262343,4,525.28
4,Ma Selena,7925940683079,6,487.52
5,De Croo Danka,8002324431175,3,464.64
6,Nasser Noor,6517814984883,1,389.00
7,Guinard Clémence,7439659172167,8,377.60
8,staniszewski claudia,7156006355271,2,358.00
9,leen chloe,7801196872007,4,350.18
